In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import os

from munch import munchify
from cafo_iowa.utils.strata import generate_stratified_sample

In [ ]:
# load config
with open("cafo_iowa/config_old.yaml", "r") as file:
    config = munchify(yaml.safe_load(file))

In [ ]:
# load data
df = pd.read_csv(config.permits.output_path)
naip_shp = gpd.read_feather(
    os.path.join(config.naip_shp.output_path, "03_quartered_tiles.feather")
)
urban_areas = gpd.read_feather(config.urban_shp.output_path)

In [ ]:
# subset dataset to only include permits with at least one swine
df = df[df["swine_animal_units"] > 0]
print(df.shape)

In [ ]:
# add a column for whether the tile is urban
urban_areas = urban_areas.to_crs(naip_shp.crs)
urban_areas["is_urban"] = True
urban_areas["urban_area"] = urban_areas.area
# NOTE: urban area uses quartered buffer tiles, so the area can be up to 1.2x the area of the original tile

# add column indicating share of urban area in given tile
naip_shp = naip_shp.merge(
    urban_areas[["qt_tile_id", "is_urban", "urban_area"]], on="qt_tile_id", how="left"
)
naip_shp["share_urban"] = naip_shp["urban_area"] / naip_shp.area
naip_shp["is_urban"] = naip_shp["is_urban"].fillna(False)
naip_shp["share_urban"] = naip_shp["share_urban"].fillna(0)

In [ ]:
# join the permit data with the naip shapefile
df = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
df.set_crs(epsg=4326, inplace=True)
df = df.to_crs(naip_shp.crs)
gdf = gpd.sjoin(naip_shp, df, op="contains", how="left")

# set epsg to 4326, for labeling checking using google earth
gdf.to_crs(epsg=4326, inplace=True)

In [ ]:
# combine the permit data with the naip shapefile, and calculate the number of permits per tile
gdf_qt = (
    gdf.groupby("qt_tile_id")
    .agg(
        animal_units=("animal_units", "sum"),
        swine_animal_units=("swine_animal_units", "sum"),
        permits=("facilityid", "count"),
        facilityid=("facilityid", "unique"),
        is_urban=("is_urban", "first"),
        share_urban=("share_urban", "sum"),
        geometry=("geometry", "first"),
    )
    .reset_index()
    .assign(
        animal_units_per_permit=lambda x: x["animal_units"] / x["permits"],
        swine_animal_units_per_permit=lambda x: x["swine_animal_units"] / x["permits"],
    )
    .fillna(0)
)

In [ ]:
# split the data into 10 quantiles
gdf_qt["animal_units_per_permit_q"] = pd.cut(
    gdf_qt["animal_units_per_permit"],
    bins=[-1, 0, 399, 499, 699, 900, 999, 1199, 1499, 1899, 1999, 2499, np.inf],
    labels=[
        "0",
        "1-399",
        "400-499",
        "500-699",
        "700-899",
        "900-999",
        "1000-1199",
        "1200-1499",
        "1500-1899",
        "1900-1999",
        "2000-2499",
        "2500+",
    ],
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=gdf_qt,
    x="animal_units_per_permit",
    hue="animal_units_per_permit_q",
    multiple="stack",
    bins=50,
)
plt.xlabel("Animal units per permit")
plt.ylabel("Frequency")
plt.title("Histogram of animal units per permit per tile")
plt.show()

In [ ]:
# create a sample dictionary specifying the proportion of each strata to sample
sample_dict_batch1 = {
    "0": 0.01,
    "700-899": 0.10,
    "900-999": 0.40,
    "1000-1199": 0.40,
    "1200-1499": 0.10,
}

batch1 = generate_stratified_sample(
    gdf_qt,
    strata_column="animal_units_per_permit_q",
    sample_size=1000,
    sampling_dict=sample_dict_batch1,
    random_state=config.random_seed,
)

# make sure that all ids in batch1 are unique
assert batch1["qt_tile_id"].nunique() == batch1.shape[0]

# add batch column to data
gdf_qt["batch"] = "No batch"
gdf_qt.loc[gdf_qt["qt_tile_id"].isin(batch1["qt_tile_id"]), "batch"] = "Batch 1"

# Add batch column to permit data
df["batch_flag"] = "No batch"

# count the number of facilities and make sure they are unique
batch1_facilities = batch1["facilityid"].explode().dropna().astype(int)

df.loc[df["facilityid"].isin(batch1_facilities), "batch_flag"] = "Batch 1"

assert batch1_facilities.unique().shape[0] == batch1_facilities.shape[0]

In [ ]:
print(
    "Batch 1\nSample size: ",
    batch1.shape[0],
    "\nN Permits: ",
    df.loc[df["batch_flag"] == "Batch 1"].shape[0],
)

In [ ]:
# check if batch1 already exists in directory and if so, check if it is the same as the current batch1
batch1 = gpd.GeoDataFrame(batch1)
existing_files = os.listdir("data/labeling/")
existing_files = [file for file in existing_files if "batch1" in file]
if existing_files:
    lastest_file = max(existing_files)
    existing_batch1 = gpd.read_feather(f"data/labeling/{lastest_file}")
    if not batch1.equals(existing_batch1):
        print(
            "The current batch1 is not the same as the existing batch1. Do you want to save batch1 to a new file?"
        )
        overwrite = input("y/n: ")
        if overwrite == "y":
            today = pd.Timestamp.today().strftime("%Y-%m-%d")
            fn = f"data/labeling/batch1_{today}.feather"
            batch1.to_feather(fn)
            print(f"The new batch1 has been saved at {fn}")
        else:
            print("The current batch1 has not been saved")
    else:
        print("The current batch1 is the same as the existing batch1. Skipping saving")
else:
    today = pd.Timestamp.today().strftime("%Y-%m-%d")
    fn = f"data/labeling/batch1_{today}.feather"
    batch1.to_feather(fn)
    print(f"The new batch1 has been saved at {fn}")

In [ ]:
def plot_histograms(batch_data, permit_data, batch_name):
    plt.figure(figsize=(20, 6))

    # First subplot
    plt.subplot(1, 3, 1)
    sns.histplot(
        data=gdf_qt,
        x="animal_units_per_permit",
        hue="batch",
        multiple="stack",
        bins=50,
    )
    plt.xlabel("Animal units per permit")
    plt.ylabel("Frequency")
    plt.title("Animal units per permit per tile")

    # Second subplot
    plt.subplot(1, 3, 2)
    sns.histplot(
        data=batch_data,
        x="animal_units_per_permit",
        hue="animal_units_per_permit_q",
        multiple="stack",
        bins=50,
    )
    plt.xlabel("Animal units per permit per tile")
    plt.ylabel("Frequency")
    plt.title(f"{batch_name}: Animal units per permit per tile")

    # Third subplot
    plt.subplot(1, 3, 3)
    sns.histplot(
        data=permit_data,
        x="animal_units",
        hue="batch_flag",
        multiple="stack",
        bins=100,
    )
    plt.xlabel("Animal units")
    plt.ylabel("Frequency")
    plt.title("Histogram of animal units per permit")

    # Adjust spacing between subplots
    plt.tight_layout()

    plt.show()

In [ ]:
plot_histograms(batch1, df, "Batch 1")

In [ ]:
# check how many obsrvations are left in each stratum
strata = gdf_qt.groupby(["animal_units_per_permit_q", "batch"]).size()
strata = strata.reset_index()
strata

In [ ]:
# Batch 2
sample_dict_batch2 = {
    "0": 0.01,
    "700-899": 0.10,
    "900-999": 0.4,
    "1000-1199": 0.05,
    "1200-1499": 0.20,
    "1500-1899": 0.20,
}

batch2 = generate_stratified_sample(
    # sample from remaining strata, where share_urban is less than 0.8. Since urban area includes buffer, we use 0.95, sicne 0.95/1.2 = 0.8
    gdf_qt[(gdf_qt["batch"] == "No batch") & (gdf_qt["share_urban"] <= 0.95)],
    strata_column="animal_units_per_permit_q",
    sample_size=1000,
    sampling_dict=sample_dict_batch2,
    random_state=config.random_seed,
)

# make sure that all ids in batch1 are unique
assert batch2["qt_tile_id"].nunique() == batch2.shape[0]

# add batch 2 to gdf
gdf_qt.loc[gdf_qt["qt_tile_id"].isin(batch2["qt_tile_id"]), "batch"] = "Batch 2"

# count the number of facilities and make sure they are unique
batch2_facilities = batch2["facilityid"].explode().dropna().astype(int)

df.loc[df["facilityid"].isin(batch2_facilities), "batch_flag"] = "Batch 2"


batch2.shape
assert batch2_facilities.unique().shape[0] == batch2_facilities.shape[0]

In [ ]:
print(
    "Batch 1\nSample size: ",
    batch1.shape[0],
    "\nN Permits: ",
    df.loc[df["batch_flag"] == "Batch 2"].shape[0],
)

In [ ]:
# check how many obsrvations are left in each stratum
strata = gdf_qt.groupby(["animal_units_per_permit_q", "batch"]).size()
strata = strata.reset_index()
strata

In [ ]:
plot_histograms(batch2, df, "Batch 2")

In [ ]:
gdf_geo = gpd.GeoDataFrame(gdf_qt)
# plot map of the data
plt.figure(figsize=(10, 6))
gdf_geo.plot(column="batch", legend=True)
plt.title("Map of the data")
plt.axis("off")
plt.show()
# save map with high resolution
gdf_geo.plot(column="batch", legend=True)
plt.title("Map of the data")
plt.axis("off")
plt.savefig("map.png", dpi=300)

In [ ]:
# check if batch2 already exists in directory and if so, check if it is the same as the current batch2
batch2 = gpd.GeoDataFrame(batch2)
existing_files = os.listdir("data/labeling/")
existing_files = [file for file in existing_files if "batch2" in file]
if existing_files:
    lastest_file = max(existing_files)
    existing_batch2 = gpd.read_feather(f"data/labeling/{lastest_file}")
    if not batch2.equals(existing_batch2):
        print(
            "The current batch2 is not the same as the existing batch2. Do you want to save batch1 to a new file?"
        )
        overwrite = input("y/n: ")
        if overwrite == "y":
            today = pd.Timestamp.today().strftime("%Y-%m-%d")
            fn = f"data/labeling/batch2_{today}.feather"
            batch2.to_feather(fn)
            print(f"The new batch2 has been saved at {fn}")
        else:
            print("The current batch2 has not been saved")
    else:
        print("The current batch2 is the same as the existing batch2. Skipping saving")
else:
    today = pd.Timestamp.today().strftime("%Y-%m-%d")
    fn = f"data/labeling/batch2_{today}.feather"
    batch2.to_feather(fn)
    print(f"The new batch2 has been saved at {fn}")